In [1]:
pip install polars

StatementMeta(, c26b79ec-b402-474a-9d04-4e80ddcc6fbf, 3, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.7/828.7 kB 9.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 75.8 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import polars as pl

StatementMeta(, c26b79ec-b402-474a-9d04-4e80ddcc6fbf, 4, Finished, Available, Finished, False)

In [3]:
N = 10_000_000 

np.random.seed(0)
airlines = np.random.choice(
    ["AA", "DL", "UA", "WN", "AS", "B6"], size=N
)

dep_delay = np.random.randint(-20, 300, size=N)
arr_delay = dep_delay + np.random.randint(-15, 15, size=N)

StatementMeta(, c26b79ec-b402-474a-9d04-4e80ddcc6fbf, 5, Finished, Available, Finished, False)

In [4]:
df_pd = pd.DataFrame({
    "Airline": airlines,
    "DepDelay": dep_delay,
    "ArrDelay": arr_delay,
})

df_pl = pl.DataFrame({
    "Airline": airlines,
    "DepDelay": dep_delay,
    "ArrDelay": arr_delay,
})

StatementMeta(, c26b79ec-b402-474a-9d04-4e80ddcc6fbf, 6, Finished, Available, Finished, False)

In [5]:
%timeit df_pd[df_pd["DepDelay"] > 10].groupby("Airline").agg(dep_mean=("DepDelay","mean"), arr_mean=("ArrDelay","mean"))

StatementMeta(, c26b79ec-b402-474a-9d04-4e80ddcc6fbf, 7, Finished, Available, Finished, False)

715 ms ± 5.97 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
%timeit df_pl.filter(pl.col("DepDelay") > 10).group_by("Airline").agg([pl.col("DepDelay").mean().alias("dep_mean"), pl.col("ArrDelay").mean().alias("arr_mean")])

StatementMeta(, c26b79ec-b402-474a-9d04-4e80ddcc6fbf, 8, Finished, Available, Finished, False)

68.3 ms ± 746 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
#Pandas takes about one second, Polars takes about a tenth of a second. That’s roughly a 10× speed‑up for the same computation.

**Polars Lazy and Eager Execution**

In [7]:
import polars as pl

df = pl.DataFrame({
    "Airline": ["AA", "DL", "AA", "UA", "DL"],
    "DepDelay": [5, 20, 60, -3, 45],
    "ArrDelay": [0, 15, 55, -10, 40],
})

StatementMeta(, c26b79ec-b402-474a-9d04-4e80ddcc6fbf, 9, Finished, Available, Finished, False)

**Eager Execution**

In [8]:
filtered = df.filter(pl.col("DepDelay") > 10)
grouped = filtered.group_by("Airline")
result_eager = grouped.agg(
    pl.col("DepDelay").mean(),
    pl.col("ArrDelay").mean()
)

result_eager

StatementMeta(, c26b79ec-b402-474a-9d04-4e80ddcc6fbf, 10, Finished, Available, Finished, False)

Airline,DepDelay,ArrDelay
str,f64,f64
"""DL""",32.5,27.5
"""AA""",60.0,55.0


**Lazy Execution**

In [9]:
lazy_query = (
    df.lazy()
      .filter(pl.col("DepDelay") > 10)
      .group_by("Airline")
      .agg(
          pl.col("DepDelay").mean(),
          pl.col("ArrDelay").mean()
      )
)

#lazy_query.collect()

StatementMeta(, c26b79ec-b402-474a-9d04-4e80ddcc6fbf, 11, Finished, Available, Finished, False)